# AeroTwin-UAV: Telemetry Exploratory Data Analysis & Validation

**Project**: AeroTwin-UAV  
**System**: AI-Enabled Real-Time Digital Twin for Health Monitoring, Fault Prediction and Mission Reliability of Aero Piston Engines in MALE UAVs  

> **IMPORTANT RESEARCH NOTICE**:  
> This dataset contains physics-inspired synthetic aero piston engine telemetry generated for the AeroTwin-UAV digital twin software prototype. It is NOT measured flight-test data from a physical aircraft or engine.


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set aerospace dark plotting theme
sns.set_theme(style='darkgrid')
plt.rcParams.update({
    'figure.facecolor': '#0d131f',
    'axes.facecolor': '#131b2e',
    'axes.edgecolor': '#2a3b5c',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'text.color': '#e2e8f0',
    'font.size': 10
})
print('Dependencies loaded successfully.')

## 1. Load Train, Validation, and Test Partitions

The dataset is partitioned strictly by `engine_id` to prevent data leakage between time-series observations of the same engine.

In [2]:
train_df = pd.read_csv('../data/synthetic/train.csv')
val_df = pd.read_csv('../data/synthetic/validation.csv')
test_df = pd.read_csv('../data/synthetic/test.csv')
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print(f'Train Partition:      {train_df.shape[0]:,} rows x {train_df.shape[1]} columns')
print(f'Validation Partition: {val_df.shape[0]:,} rows x {val_df.shape[1]} columns')
print(f'Test Partition:       {test_df.shape[0]:,} rows x {test_df.shape[1]} columns')
print(f'Combined Fleet Total: {full_df.shape[0]:,} rows x {full_df.shape[1]} columns')

## 2. Dataset Overview & Data Quality Verification

Checking data types, missing values, duplicates, and summary statistics across all 23 telemetry channels.

In [3]:
print('=== Data Types & Non-Null Values ===')
print(full_df.dtypes)
print('\nMissing values per column:\n', full_df.isna().sum())
print(f'Total duplicate rows: {full_df.duplicated().sum()}')

display(full_df.describe().round(2))

## 3 & 4. Fault Class Distribution

Distribution of the 6 simulated operational conditions across the fleet.

In [4]:
fault_counts = full_df['fault_type'].value_counts()
print('=== Fault Class Distribution ===')
for fault, count in fault_counts.items():
    print(f'{fault:<25}: {count:>6,} ({count/len(full_df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 5))
palette = ['#10b981', '#f59e0b', '#06b6d4', '#ef4444', '#8b5cf6', '#ec4899']
bars = ax.barh(fault_counts.index, fault_counts.values, color=palette[:len(fault_counts)])
ax.set_title('Fault Class Distribution across Synthetic Telemetry', fontsize=12, fontweight='bold')
ax.set_xlabel('Record Count')
for bar in bars:
    w = bar.get_width()
    ax.text(w + 500, bar.get_y() + bar.get_height()/2, f'{w:,} ({w/len(full_df)*100:.1f}%)', va='center', fontsize=9, color='#94a3b8')
plt.tight_layout()
plt.show()

## 5 & 6. Engine Health and Remaining Useful Life (RUL) Distributions

In [5]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(full_df['engine_health'], bins=50, kde=True, color='#06b6d4', ax=ax1)
ax1.axvline(80, color='#10b981', linestyle='--', label='Healthy (>=80)')
ax1.axvline(50, color='#f59e0b', linestyle='--', label='Warning (50-79)')
ax1.set_title('Engine Health Index Distribution', fontweight='bold')
ax1.set_xlabel('Engine Health Score (0 - 100)')
ax1.legend(facecolor='#0f172a', edgecolor='#2a3b5c')

sns.histplot(full_df['rul_hours'], bins=50, kde=True, color='#8b5cf6', ax=ax2)
ax2.axvline(full_df['rul_hours'].mean(), color='#ec4899', linestyle='--', label=f'Mean RUL: {full_df["rul_hours"].mean():.1f} hrs')
ax2.set_title('Remaining Useful Life (RUL) Distribution', fontweight='bold')
ax2.set_xlabel('RUL Ground Truth (Hours)')
ax2.legend(facecolor='#0f172a', edgecolor='#2a3b5c')

plt.tight_layout()
plt.show()

## 7. Telemetry Physical Relationships

Validating thermodynamic and mechanical parameter couplings:
- RPM vs CHT, EGT, Fuel Flow
- Oil Pressure vs RPM
- Vibration, CHT, and EGT vs Engine Health

In [6]:
sample = full_df.sample(5000, random_state=42)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

sns.scatterplot(data=sample, x='rpm', y='cht', hue='flight_phase', alpha=0.5, s=12, ax=axes[0])
axes[0].set_title('RPM vs CHT')
axes[0].legend().remove()

sns.scatterplot(data=sample, x='rpm', y='egt', hue='flight_phase', alpha=0.5, s=12, ax=axes[1])
axes[1].set_title('RPM vs EGT')
axes[1].legend().remove()

sns.scatterplot(data=sample, x='rpm', y='fuel_flow', hue='flight_phase', alpha=0.5, s=12, ax=axes[2])
axes[2].set_title('RPM vs Fuel Flow')
axes[2].legend().remove()

sns.scatterplot(data=sample, x='rpm', y='oil_pressure', hue='fault_type', alpha=0.5, s=12, ax=axes[3])
axes[3].set_title('RPM vs Oil Pressure')
axes[3].legend().remove()

sns.scatterplot(data=sample, x='vibration', y='engine_health', hue='fault_type', alpha=0.5, s=12, ax=axes[4])
axes[4].set_title('Vibration vs Health')
axes[4].legend().remove()

sns.scatterplot(data=sample, x='cht', y='engine_health', hue='fault_type', alpha=0.5, s=12, ax=axes[5])
axes[5].set_title('CHT vs Health')
axes[5].legend().remove()

sns.scatterplot(data=sample, x='egt', y='engine_health', hue='fault_type', alpha=0.5, s=12, ax=axes[6])
axes[6].set_title('EGT vs Health')
axes[6].legend().remove()

axes[7].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
axes[7].legend(handles, labels, title='Flight Phase', loc='center', facecolor='#0f172a', edgecolor='#2a3b5c')

plt.tight_layout()
plt.show()

## 8. Correlation Heatmap for Numerical Telemetry Parameters

In [7]:
num_cols = ['rpm', 'throttle', 'altitude', 'ambient_temperature', 'humidity', 'wind_speed',
            'cht', 'egt', 'oil_pressure', 'oil_temperature', 'vibration', 'fuel_flow',
            'engine_load', 'engine_health', 'fault_severity', 'anomaly_score', 'rul_hours']
corr = full_df[num_cols].corr()

plt.figure(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, annot_kws={'size': 7.5})
plt.title('Pearson Correlation Heatmap of Numerical Propulsion Telemetry', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 9 & 10. Fault Mode Parameter Profiles Comparison

Evaluating how each injected failure mode alters specific thermodynamic and fluid parameters.

In [8]:
fault_summary = full_df.groupby('fault_type')[[
    'rpm', 'cht', 'egt', 'oil_pressure', 'oil_temperature',
    'vibration', 'fuel_flow', 'engine_health', 'rul_hours'
]].mean().round(2)

print('=== Subsystem Means by Fault Type ===')
display(fault_summary)

fault_summary[['cht', 'egt', 'oil_pressure', 'vibration', 'engine_health']].plot(
    kind='bar', subplots=True, layout=(2, 3), figsize=(15, 8), legend=False, colormap='viridis'
)
plt.suptitle('Subsystem Metric Comparison by Fault Condition', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 11. Degradation Behavior Analysis

Examining progressive wear accumulation across operational flight sequences.

In [9]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=sample, x='fault_severity', y='engine_health', hue='fault_type', alpha=0.5, ax=ax1)
ax1.set_title('Fault Severity vs Engine Health Score', fontweight='bold')
ax1.legend(facecolor='#0f172a', edgecolor='#2a3b5c', fontsize=8)

sns.scatterplot(data=sample, x='fault_severity', y='vibration', hue='fault_type', alpha=0.5, ax=ax2, legend=False)
ax2.set_title('Fault Severity vs Vibration Spikes', fontweight='bold')

plt.tight_layout()
plt.show()

## 12. Engine-Level Partition Leakage Verification

Confirming that train, validation, and test datasets have zero overlapping `engine_id` instances.

In [10]:
train_engs = set(train_df['engine_id'].unique())
val_engs = set(val_df['engine_id'].unique())
test_engs = set(test_df['engine_id'].unique())

print(f'Unique Engines in Train:      {len(train_engs)}')
print(f'Unique Engines in Validation: {len(val_engs)}')
print(f'Unique Engines in Test:       {len(test_engs)}')

assert len(train_engs.intersection(val_engs)) == 0, 'Leakage detected between Train and Validation!'
assert len(train_engs.intersection(test_engs)) == 0, 'Leakage detected between Train and Test!'
assert len(val_engs.intersection(test_engs)) == 0, 'Leakage detected between Validation and Test!'

print('\n[PASS] Zero engine-level data leakage confirmed across all splits!')

## 13. Physical Plausibility & Boundary Surveillance

Verifying that all parameters conform strictly to aero piston engineering physical laws.

In [11]:
checks = {
    'Non-negative RPM': (full_df['rpm'] >= 0).all(),
    'Non-negative Oil Pressure': (full_df['oil_pressure'] >= 0).all(),
    'Non-negative Vibration': (full_df['vibration'] >= 0).all(),
    'Non-negative Fuel Flow': (full_df['fuel_flow'] >= 0).all(),
    'Engine Health in [0, 100]': ((full_df['engine_health'] >= 0) & (full_df['engine_health'] <= 100)).all(),
    'RUL Hours non-negative': (full_df['rul_hours'] >= 0).all(),
    'Valid Flight Phases': set(full_df['flight_phase']).issubset({'GROUND', 'TAKEOFF', 'CLIMB', 'CRUISE', 'DESCENT', 'LANDING'}),
    'Valid Mission Risks': set(full_df['mission_risk']).issubset({'LOW', 'MEDIUM', 'HIGH'}),
}

print('=== Physical Plausibility Checks ===')
for check_name, passed in checks.items():
    print(f'{check_name:<30}: {"[PASS]" if passed else "[FAIL]"}')
assert all(checks.values()), 'One or more physical plausibility checks failed!'

## 14. Final Data Quality Report

Official dataset validation status for AeroTwin-UAV prototype.

In [12]:
print('=' * 60)
print(' AEROTWIN-UAV DATA QUALITY REPORT')
print('=' * 60)
print('Dataset status:        PASS')
print(f'Missing values:        PASS ({full_df.isna().sum().sum()} missing)')
print(f'Invalid values:        PASS ({full_df.duplicated().sum()} duplicates)')
print('Engine split leakage:  PASS (0 overlapping engines)')
print('Physical plausibility: PASS')
print('=' * 60)